# 33. TCO and Cost Model | 算力评估与 TCO 模型

**难度：** Medium | **环境：** CPU-first | **标签：** `系统工程`, `成本评估`, `TCO` | **目标人群：** 硬件约束学习者

![总拥有成本决策链](../docs/public/01_Hardware_Math_and_Systems/33_tco_decision_chain.svg)

---

## 本节导读

算力采购价格只覆盖设备进入系统时的一次成本。长期运行还会产生能耗、运维、迁移、停机和低利用率成本，并受到折旧周期影响。

本节建立“固定成本与运行成本 → 利用率与风险 → 单位成功请求成本”的账本。完成后，你应能解释低采购价为什么不一定带来低单位成本，并为部署与性能候选提供统一成本口径。

**关键词：** 总拥有成本（TCO）、单位成本、能耗

---

## 前置阅读

**导语：** 这一页先把编译、选型和成本判断放到同一条判断链里，再看 TCO 为什么不能只看单卡报价。

- [09. AI Compilers and Graph Optimization | AI 编译器与计算图优化](./09_AI_Compilers_and_Graph_Optimization.ipynb)
- [10. AI Chips Overview and Alternatives | 算力现状与替代方案](./10_Domestic_AI_Chips_Overview.ipynb)
- [32. TVM / MLIR Deep Practice | TVM / MLIR 深度实践](./32_TVM_MLIR_Deep_Practice.ipynb)


## Q1：TCO 为什么比单卡报价更重要？


TCO 看的是总拥有成本，不只是买卡的钱。

它会把采购、能耗、运维、迁移、停机风险和替换成本都放进来。对于长期运行的系统来说，单卡便宜不一定代表整体便宜，因为后续的电费、维护和迁移可能更贵。

所以真正要比较的是一段时间内谁更能稳定地把业务跑起来。


### Q1 成本账本：采购、能耗、运维和迁移分别是多少

看总账，而不是只看首付。

In [ ]:
def tco_ledger(purchase_cost, power_kw, electricity_per_kwh, ops_cost_per_month, migration_cost, months, utilization):
    """按统一货币单位计算持有期内采购、能耗、运维和迁移成本。"""
    if min(purchase_cost, power_kw, electricity_per_kwh, ops_cost_per_month, migration_cost, months) < 0:
        raise ValueError('cost inputs must be non-negative')
    if months <= 0 or not 0 <= utilization <= 1:
        raise ValueError('invalid months or utilization')
    powered_hours = months * 30 * 24 * utilization
    energy_cost = power_kw * powered_hours * electricity_per_kwh
    components = {'purchase_cost': purchase_cost, 'energy_cost': energy_cost, 'operations_cost': ops_cost_per_month * months, 'migration_cost': migration_cost}
    return {'components': components, 'total_cost': sum(components.values()), 'powered_hours': powered_hours}

gpu_a = tco_ledger(30_000, 0.7, 0.8, 200, 2_000, months=24, utilization=0.7)
gpu_b = tco_ledger(22_000, 0.9, 0.8, 350, 8_000, months=24, utilization=0.7)
assert gpu_a['total_cost'] == sum(gpu_a['components'].values())
print('A:', gpu_a)
print('B:', gpu_b)

## Q2：为什么迁移成本和风险要算进模型里？


硬件切换不只是换设备，还会带来软件适配、调试、重构和性能回归风险。

如果迁移需要重新适配编译器、kernel、通信库或部署流程，那这些工作本身就是成本的一部分。

所以 TCO 不是财务问题和技术问题的二选一，而是把两者放进同一个判断框架。


### Q2 风险账本：迁移工程量与失败代价如何计入

适配成本本身就是成本。

In [ ]:
def migration_cost(engineering_hours, hourly_rate, validation_hours, downtime_hours, downtime_cost_per_hour, rollback_probability):
    """计算迁移工程成本和按回滚概率加权的停机风险成本。"""
    if min(engineering_hours, hourly_rate, validation_hours, downtime_hours, downtime_cost_per_hour) < 0:
        raise ValueError('migration inputs must be non-negative')
    if not 0 <= rollback_probability <= 1:
        raise ValueError('rollback_probability must be in [0, 1]')
    labor_cost = (engineering_hours + validation_hours) * hourly_rate
    expected_rollback_cost = downtime_hours * downtime_cost_per_hour * rollback_probability
    return {'labor_cost': labor_cost, 'expected_rollback_cost': expected_rollback_cost, 'total_cost': labor_cost + expected_rollback_cost}

low_risk = migration_cost(120, 100, 40, 2, 5_000, 0.05)
high_risk = migration_cost(120, 100, 40, 2, 5_000, 0.5)
assert high_risk['total_cost'] > low_risk['total_cost']
print('low risk:', low_risk)
print('high risk:', high_risk)

## Q3：怎样用成本模型做更合理的选型？


做选型时，最好把成本、风险和收益放到同一个表里。

常见比较维度包括：
- 初始采购成本；
- 运行时电力和运维成本；
- 软件栈成熟度；
- 迁移和维护风险；
- 未来扩展的稳定性。

这样得到的不是“最低价”，而是“最适合当前约束的方案”。


### Q3 单位成本：每百万成功请求承担多少 TCO

先看长期约束，再看短期价格。

In [ ]:
def cost_per_million_successful_requests(total_cost, requests_per_second, availability, months):
    """用成功请求数归一化持有期成本。"""
    if total_cost < 0 or requests_per_second <= 0 or months <= 0 or not 0 < availability <= 1:
        raise ValueError('invalid cost, throughput, availability or months')
    successful_requests = requests_per_second * availability * months * 30 * 24 * 3600
    return total_cost / successful_requests * 1_000_000

options = {
    'stable_gpu': cost_per_million_successful_requests(gpu_a['total_cost'], 20, 0.99, 24),
    'alternative': cost_per_million_successful_requests(gpu_b['total_cost'], 18, 0.96, 24),
}
assert all(value > 0 for value in options.values())
print(options)
print('lower unit cost:', min(options, key=options.get))

## Q4：为什么扩容节奏和折旧周期也会改变 TCO 结论？


TCO 不只是“买得起还是用得起”，还取决于你要在多长时间内把这套系统跑满。

如果扩容太慢，前期买来的算力可能闲置，单位产出成本会被拉高；如果折旧周期太短，硬件还没充分发挥就要更新，账面成本也会迅速上升。

因此真正的成本模型，往往要同时考虑：
- 初始采购成本；
- 运行成本；
- 迁移和切换成本；
- 扩容与折旧节奏。

把这些变量放在一起看，才更接近真实的技术选型决策。


### Q4 利用率对照：折旧期内每个有效使用小时承担多少成本

In [ ]:
def effective_cost_per_used_hour(purchase_cost, operations_cost, migration_cost, horizon_months, depreciation_months, utilization):
    """按折旧期分摊采购成本，并以实际使用小时归一化。"""
    if min(purchase_cost, operations_cost, migration_cost) < 0:
        raise ValueError('costs must be non-negative')
    if horizon_months <= 0 or depreciation_months <= 0 or not 0 < utilization <= 1:
        raise ValueError('invalid horizon, depreciation or utilization')
    allocated_purchase = purchase_cost * min(horizon_months / depreciation_months, 1.0)
    used_hours = horizon_months * 30 * 24 * utilization
    total_allocated_cost = allocated_purchase + operations_cost + migration_cost
    return {'allocated_purchase': allocated_purchase, 'used_hours': used_hours, 'cost_per_used_hour': total_allocated_cost / used_hours}

scenarios = {
    'high_utilization': effective_cost_per_used_hour(30_000, 5_000, 2_000, 24, 36, 0.8),
    'low_utilization': effective_cost_per_used_hour(30_000, 5_000, 2_000, 24, 36, 0.3),
}
assert scenarios['high_utilization']['cost_per_used_hour'] < scenarios['low_utilization']['cost_per_used_hour']
print(scenarios)


## ⚠️ 常见误区

- 只看采购价会漏掉长期成本。
- 迁移和运维不是附带项，它们会真实影响 TCO。
- 选型不是找最便宜，而是找最适合约束的方案。
- 成本模型应同时覆盖技术和业务风险。

## 相关阅读

**导语：** 如果想继续把成本、动态 shape 和虚拟化带来的长期影响串起来，可以接着看这些页。

- [29. CUDA Stream Advanced Scheduling | CUDA Stream 高级调度](./29_CUDA_Stream_Advanced_Scheduling.ipynb)
- [30. Dynamic Shape Handling | 动态 Shape 处理](./30_Dynamic_Shape_Handling.ipynb)
- [31. GPU Virtualization and MIG | GPU 虚拟化与 MIG](./31_GPU_Virtualization_and_MIG.ipynb)
---